# Table 7 — accuracy saturation and the Rc-PGM advantage conditions

This notebook regenerates Table 7 of the paper
*Computational Complexity Analysis of Quantum-Inspired Pretty Good Measurement Classifiers*.

For each of the eleven datasets it sweeps the copy number $c = 1, \dots, 9$, fits the
k-PGM on a stratified 80/20 split and records the test-set accuracy. The row reported in
the table is the one at which the accuracy is maximal; ties are resolved in favour of the
smallest $c$.

At that $c$ the three advantage conditions of Subsection *Rc-PGM versus k-PGM* are
evaluated:

| condition | inequality |
|---|---|
| training time | $N > \sqrt[3]{l}\, d_{\mathrm{sym}}$ |
| training memory | $N > d_{\mathrm{sym}}^{2}$ |
| prediction time **and** memory | $N > l\, d_{\mathrm{sym}}^{2} / (d + r_{G^{c}})$ |

**Encoded dimension.** As everywhere in the paper, $d$ is the dimension of the *encoded*
space, not the number of raw features: the `amplit` encoding appends one component to each
vector before $\ell_2$-normalisation, so a dataset with $p$ raw features gives $d = p + 1$,
and the symmetric subspace has dimension

$$d_{\mathrm{sym}} = \binom{d + c - 1}{c}.$$

All the conditions are evaluated in this encoded space; the convention also guarantees the
consistency bound $r_{G^{c}} \le \min(N, d_{\mathrm{sym}})$ in every row, where
$r_{G^{c}} = \operatorname{rank}(G^{c})$ is the numerical rank of the *$c$-th power* Gram
matrix — not of $G$ — retained by the low-rank spectral decomposition of the k-PGM
(eigenvalue threshold $10^{-6}$). In the code below it is the variable `r_Gc`.

**Prediction condition.** The full prediction cost of the k-PGM is
$\mathcal{O}\bigl(N(d + r_{G^{c}})\bigr)$ in both time and memory, so a single threshold
governs the two prediction metrics.

**Accuracy.** The accuracy column is shared by the three classifiers: c-PGM, k-PGM and
Rc-PGM are equivalent in classification performance, so it is computed from the k-PGM
alone and only serves to locate the saturation point $c$.

In [ ]:
import math
import os
from glob import glob
from pathlib import Path

import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Run from the repository root, so that the relative dataset paths resolve
# whether the notebook is launched from `notebooks/` or from the root.
if Path("datasets").is_dir() is False and Path("../datasets").is_dir():
    os.chdir("..")

# The datasets are not carried in the repository.
if not sorted(glob("datasets/*.csv")):
    raise SystemExit(
        "datasets/ is empty: run `python scripts/fetch_datasets.py` from the "
        "repository root to download and rebuild the eleven CSV files."
    )

from qunica.classifiers.KPGMC_Low_Rank import KPGM

# --- selection protocol, in full ---------------------------------------
# The sweep is exhaustive over a fixed grid: EVERY dataset is fitted at
# c = 1, 2, ..., MAX_COPIES, with no early stopping and no per-dataset range.
# The row reported in Table 7 is the SMALLEST c attaining the maximum
# test-set accuracy on that grid (`idxmax` keeps the first occurrence).
# It is an argmax over the grid, not a plateau-detection rule: where the
# accuracy is flat, the smallest c of the plateau is the one reported.
# The accuracies come from ONE seeded stratified 80/20 split -- the same
# split `run_benchmarks.py` uses, which is why Component A reproduces them
# exactly -- and are therefore single-split point estimates, with no
# repetition and no confidence interval.
ENCODING = "amplit"
DTYPE = torch.float64
TEST_SIZE = 0.2
RANDOM_STATE = 42
MAX_COPIES = 9

print("working directory:", Path.cwd())

## Sweep over the datasets and the copy number

In [ ]:
records = []

# Only the CSV files: Skin_NonSkin.txt is used by the benchmark harness, not by Table 7.
dataset_paths = sorted(glob("datasets/*.csv"))

for path in dataset_paths:
    name = Path(path).stem
    frame = pd.read_csv(path, header=None)

    train, test = train_test_split(
        frame,
        test_size=TEST_SIZE,
        shuffle=True,
        stratify=frame.iloc[:, -1],
        random_state=RANDOM_STATE,
    )

    X_train, y_train = train.iloc[:, :-1].to_numpy(), train.iloc[:, -1].to_numpy()
    X_test, y_test = test.iloc[:, :-1].to_numpy(), test.iloc[:, -1].to_numpy()

    n_samples, n_features = X_train.shape
    n_classes = len(set(y_train))

    # The `amplit` encoding appends one component before normalisation.
    d_enc = n_features + 1

    for n_copies in range(1, MAX_COPIES + 1):
        model = KPGM(n_copies=n_copies, encoding=ENCODING, dtype=DTYPE)
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))

        # Numerical rank of G^c -- NOT of G -- retained by the low-rank
        # decomposition. This is r_{G^c} of the paper.
        r_Gc = model.lam_inv_sqrt.shape[0]

        # Dimension of the symmetric subspace, in the encoded space.
        d_sym = math.comb(d_enc + n_copies - 1, n_copies)

        threshold_train_time = math.pow(n_classes, 1 / 3) * d_sym
        threshold_train_memory = d_sym**2
        threshold_prediction = n_classes * d_sym**2 / (d_enc + r_Gc)

        records.append(
            {
                "Dataset": name,
                "Samples": n_samples,
                "Features": n_features,
                "Copies": n_copies,
                "Classes": n_classes,
                "d_enc": d_enc,
                "d_sym": d_sym,
                "r_Gc": r_Gc,
                "Rc-PGM Tr Time Threshold": threshold_train_time,
                "Rc-PGM Tr Memory Threshold": threshold_train_memory,
                "Rc-PGM Pred Threshold": threshold_prediction,
                "Rc-PGM Tr Win Time": n_samples > threshold_train_time,
                "Rc-PGM Tr Win Memory": n_samples > threshold_train_memory,
                "Rc-PGM Pred Win (Time & Memory)": n_samples > threshold_prediction,
                "Accuracy": accuracy,
            }
        )

sweep = pd.DataFrame(records)
print(sweep.shape)
sweep.head()

## Selected copy number

One row per dataset: the **smallest** copy number attaining the maximum test-set accuracy
over the swept grid $c = 1,\dots,9$ (`idxmax` keeps the first occurrence). This is an
argmax over the grid, not a plateau-detection criterion; the cell after the selection
prints the whole accuracy curve, so the shape behind each choice is visible, and flags the
two situations in which the selected $c$ should not be read as a saturation point:

- $c^{*} = 1$ — tensor copies never improve the accuracy on this split;
- $c^{*} = $ `MAX_COPIES` — the maximum sits at the edge of the swept grid, so the
  accuracy may still be rising beyond it.

In [ ]:
best = (
    sweep.loc[sweep.groupby("Dataset")["Accuracy"].idxmax()]
    .sort_values("Dataset")
    .reset_index(drop=True)
)
best

In [ ]:
# The process behind the `c` column of Table 7, made visible: the whole
# accuracy curve, with the selected point starred, and the boundary flags.
pivot = sweep.pivot(index="Dataset", columns="Copies", values="Accuracy")
selected = best.set_index("Dataset")["Copies"]

print("accuracy vs c   (* = the value reported in Table 7)")
print(
    f"{'dataset':<20}"
    + "".join(f"{'c=' + str(c):>7}" for c in pivot.columns)
    + "   c*  note"
)
for name, row in pivot.iterrows():
    c_star = int(selected[name])
    cells = "".join(
        f"{('*' if c == c_star else ' ') + f'{row[c]:.3f}':>7}" for c in pivot.columns
    )
    note = []
    if c_star == 1:
        note.append("no gain from copies")
    elif c_star == MAX_COPIES:
        note.append("AT THE EDGE OF THE GRID: the accuracy may still be rising")
    elif (row[[c for c in pivot.columns if c >= c_star]] == row[c_star]).all():
        note.append("flat from c* onwards")
    print(f"{name:<20}{cells}   {c_star:>2}  " + "; ".join(note))

at_edge = [n for n in pivot.index if int(selected[n]) == MAX_COPIES]
if at_edge:
    subject = (
        f"{at_edge[0]} attains" if len(at_edge) == 1 else f"{', '.join(at_edge)} attain"
    )
    print(
        f"\nNOTE: {subject} the maximum at c = {MAX_COPIES}, the last value of "
        f"the grid;\nthere the reported c is a grid limit, not an observed plateau."
    )

In [ ]:
# The full sweep is deposited alongside the selected rows, so that the
# selection can be audited without re-running the notebook.
sweep.to_csv("results_table7_sweep.csv", index=False)
print(
    f"written: results_table7_sweep.csv ({len(sweep)} rows = "
    f"{sweep.Dataset.nunique()} datasets x {MAX_COPIES} copy numbers)"
)

In [ ]:
# Consistency of the encoded-space convention: r_Gc <= min(N, d_sym) must hold everywhere.
violations = sweep[sweep["r_Gc"] > sweep[["Samples", "d_sym"]].min(axis=1)]
assert violations.empty, violations
print(f"r_Gc <= min(N, d_sym) holds in all {len(sweep)} swept rows.")

In [ ]:
best.to_csv("results_table7.csv", index=False)
print("written: results_table7.csv")

## Table 7 as printed in the paper

The cell below reduces the row set to the columns of Table 7 and emits the corresponding
LaTeX body, so that it can be compared line by line with the manuscript.

In [ ]:
# The `d` column of Table 7 is the ENCODED dimension (`d_enc`), not the raw
# feature count: it is the quantity every complexity expression of the paper uses.
table7 = best.rename(
    columns={
        "Samples": "N",
        "d_enc": "d",
        "Copies": "c",
        "Classes": "l",
        "d_sym": "d_sym",
        "r_Gc": "r_Gc",
        "Accuracy": "Max. Accuracy",
        "Rc-PGM Tr Win Time": "Train time",
        "Rc-PGM Tr Win Memory": "Train memory",
        "Rc-PGM Pred Win (Time & Memory)": "Prediction",
    }
)[
    [
        "Dataset",
        "N",
        "d",
        "c",
        "l",
        "d_sym",
        "r_Gc",
        "Max. Accuracy",
        "Train time",
        "Train memory",
        "Prediction",
    ]
]
table7["Max. Accuracy"] = table7["Max. Accuracy"].round(3)
table7

In [ ]:
def latex_rows(frame):
    lines = []
    for row in frame.itertuples(index=False):
        lines.append(
            f"{row.Dataset.replace('_', ' ')} & {row.N} & {row.d} & {row.c} & {row.l} & "
            f"{row.d_sym} & {row.r_Gc} & {row[7]:.3f} & "
            f"{row[8]} & {row[9]} & {row[10]} \\\\"
        )
    return "\n".join(lines)


print(latex_rows(table7))